# Patient Readmission Risk Prediction
## 02 - Data Preprocessing

This notebook prepares the Diabetes 130-US Hospitals dataset
for machine learning models.

The target is binary classification:

1 = Readmitted within 30 days
0 = Not readmitted within 30 days

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
df = pd.read_csv("../data/raw/diabetic_data.csv")

print("Original shape:", df.shape)

Original shape: (101766, 50)


In [3]:
df_clean = df.copy()

In [4]:
df_clean["readmitted_30_days"] = (
    df_clean["readmitted"] == "<30"
).astype(int)

In [5]:
df_clean["readmitted_30_days"].value_counts()

readmitted_30_days
0    90409
1    11357
Name: count, dtype: int64

In [6]:
df_clean["readmitted_30_days"].value_counts(normalize=True).mul(100).round(2)

readmitted_30_days
0    88.84
1    11.16
Name: proportion, dtype: float64

In [7]:
df_clean = df_clean.replace("?", np.nan)

In [8]:
df_clean.isna().sum().sort_values(ascending=False).head(15)

weight                 98569
max_glu_serum          96420
A1Cresult              84748
medical_specialty      49949
payer_code             40256
race                    2273
diag_3                  1423
diag_2                   358
diag_1                    21
patient_nbr                0
encounter_id               0
time_in_hospital           0
admission_source_id        0
num_procedures             0
num_lab_procedures         0
dtype: int64

In [9]:
columns_to_drop = [
    "encounter_id",
    "patient_nbr",
    "weight",
    "medical_specialty",
    "payer_code",
    "readmitted"
]

df_clean = df_clean.drop(columns=columns_to_drop)

In [10]:
categorical_missing_columns = [
    "race",
    "diag_1",
    "diag_2",
    "diag_3"
]

for col in categorical_missing_columns:
    df_clean[col] = df_clean[col].fillna("Unknown")

In [11]:
df_clean.isna().sum().sort_values(ascending=False).head(20)

max_glu_serum               96420
A1Cresult                   84748
age                             0
gender                          0
race                            0
admission_source_id             0
time_in_hospital                0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
admission_type_id               0
discharge_disposition_id        0
number_inpatient                0
number_emergency                0
diag_1                          0
diag_2                          0
number_diagnoses                0
diag_3                          0
metformin                       0
dtype: int64

In [12]:
df_clean["max_glu_serum"] = df_clean["max_glu_serum"].fillna("None")
df_clean["A1Cresult"] = df_clean["A1Cresult"].fillna("None")

In [13]:
df_clean.isna().sum().sum()

np.int64(0)

In [14]:
print("Shape after cleaning:", df_clean.shape)

Shape after cleaning: (101766, 45)


In [15]:
df_clean.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_30_days
0,Caucasian,Female,[0-10),6,25,1,1,41,0,1,...,No,No,No,No,No,No,No,No,No,0
1,Caucasian,Female,[10-20),1,1,7,3,59,0,18,...,No,Up,No,No,No,No,No,Ch,Yes,0
2,AfricanAmerican,Female,[20-30),1,1,7,2,11,5,13,...,No,No,No,No,No,No,No,No,Yes,0
3,Caucasian,Male,[30-40),1,1,7,2,44,1,16,...,No,Up,No,No,No,No,No,Ch,Yes,0
4,Caucasian,Male,[40-50),1,1,7,1,51,0,8,...,No,Steady,No,No,No,No,No,Ch,Yes,0


In [16]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 45 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   race                      101766 non-null  str  
 1   gender                    101766 non-null  str  
 2   age                       101766 non-null  str  
 3   admission_type_id         101766 non-null  int64
 4   discharge_disposition_id  101766 non-null  int64
 5   admission_source_id       101766 non-null  int64
 6   time_in_hospital          101766 non-null  int64
 7   num_lab_procedures        101766 non-null  int64
 8   num_procedures            101766 non-null  int64
 9   num_medications           101766 non-null  int64
 10  number_outpatient         101766 non-null  int64
 11  number_emergency          101766 non-null  int64
 12  number_inpatient          101766 non-null  int64
 13  diag_1                    101766 non-null  str  
 14  diag_2                    10176

In [17]:
df_clean["total_previous_visits"] = (
    df_clean["number_outpatient"]
    + df_clean["number_emergency"]
    + df_clean["number_inpatient"]
)

In [18]:
def categorize_diagnosis(code):
    try:
        code = str(code)

        if code.startswith("250"):
            return "Diabetes"
        
        code_num = float(code)

        if 390 <= code_num <= 459 or code_num == 785:
            return "Circulatory"
        elif 460 <= code_num <= 519 or code_num == 786:
            return "Respiratory"
        elif 520 <= code_num <= 579 or code_num == 787:
            return "Digestive"
        elif 580 <= code_num <= 629 or code_num == 788:
            return "Genitourinary"
        elif 140 <= code_num <= 239:
            return "Neoplasms"
        elif 710 <= code_num <= 739:
            return "Musculoskeletal"
        elif 800 <= code_num <= 999:
            return "Injury"
        else:
            return "Other"

    except:
        return "Unknown"

In [19]:
for col in ["diag_1", "diag_2", "diag_3"]:
    df_clean[col] = df_clean[col].apply(categorize_diagnosis)

In [20]:
df_clean[["diag_1", "diag_2", "diag_3"]].head()

,diag_1,diag_2,diag_3
0,Diabetes,Unknown,Unknown
1,Other,Diabetes,Other
2,Other,Diabetes,Unknown
3,Other,Diabetes,Circulatory
4,Neoplasms,Neoplasms,Diabetes


In [21]:
df_clean["diag_1"].value_counts()

diag_1
Circulatory        30437
Other              16527
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Unknown             1666
Name: count, dtype: int64

In [22]:
print("Total missing values:", df_clean.isna().sum().sum())

Total missing values: 0


In [23]:
print("Final shape:", df_clean.shape)

Final shape: (101766, 46)


In [24]:
categorical_columns = df_clean.select_dtypes(include=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


C:\Users\Ganesha\AppData\Local\Temp\ipykernel_10688\2952029441.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_clean.select_dtypes(include=["object"]).columns.tolist()


In [25]:
df_encoded = pd.get_dummies(
    df_clean,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

In [26]:
print("Final shape:", df_encoded.shape)

Final shape: (101766, 112)


In [27]:
df_encoded.head()

,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,insulin_Up,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-rosiglitazone_Steady,metformin-pioglitazone_Steady,change_No,diabetesMed_Yes
0,6,25,1,1,41,0,1,0,0,0,...,0,1,0,0,0,0,0,0,1,0
1,1,1,7,3,59,0,18,0,0,0,...,1,1,0,0,0,0,0,0,0,1
2,1,1,7,2,11,5,13,2,0,1,...,0,1,0,0,0,0,0,0,1,1
3,1,1,7,2,44,1,16,0,0,0,...,1,1,0,0,0,0,0,0,0,1
4,1,1,7,1,51,0,8,0,0,0,...,0,1,0,0,0,0,0,0,0,1


In [28]:
print("Total missing values:", df_encoded.isna().sum().sum())

Total missing values: 0


In [29]:
print(df_encoded["readmitted_30_days"].value_counts())

readmitted_30_days
0    90409
1    11357
Name: count, dtype: int64


In [30]:
df_encoded.to_csv(
    "../data/processed/readmission_cleaned.csv",
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
